# Deterministic Coder: your own model server on Colab

> **EXPERIMENTAL. This notebook has not yet been verified on a live Colab runtime.** Expect to fix things. If a cell fails, the error message is the useful part.

- It serves a model for **your own interactive use**, through **your own Tailscale network**. Nothing is exposed to the public internet.
- Google's Colab terms limit what notebooks may be used for, including serving models to other machines. This is for your own use on your own Colab account; you are responsible for staying within Google's terms. **Do not share the address or the key.**
- A free runtime may disconnect when it is idle, and a paid GPU (L4, A100, H100) is **not guaranteed** to be available. Colab Pro or Pro+ is needed for those GPUs.
- **Clear outputs before sharing a copy of this notebook:** it prints a session API key.

**Which model runs, by GPU** (the cell that picks it prints what it chose; every row is *unverified*, no row has been run)

| GPU | Tier | Model | Context | Download |
|---|---|---|---|---|
| A100 or H100, 80 GB | Father | `Qwen/Qwen3.8-27B` (bf16) | 131072 tokens | about 55.6 GB |
| A100, 40 GB | Father | `Qwen/Qwen3.8-27B-FP8` (Qwen's own 8-bit repo) | 32768 tokens (65536 is unproven) | about 30.9 GB |
| L4, 24 GB | Son | `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` (4-bit) | 32768 tokens | about 10 GB |
| T4, 16 GB (free) | Son | `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` (4-bit) | 8192 tokens | about 10 GB |

The 27B model cannot fit on an L4 or a T4 (it needs a GPU of at least 38 GB), so those GPUs use the smaller 14B 4-bit model. The T4 path is **unverified**: nothing here claims it works or that it is fast.

**Before you run it (one time)**
1. Install Tailscale on the computer that runs Deterministic Coder and sign in to the same tailnet you will use here.
2. In the Tailscale admin page, create an **auth key** (Settings > Keys): tick *Reusable* and *Ephemeral*.
3. In Colab, open the key icon (Secrets) in the left bar, add a secret named `TS_AUTHKEY`, paste the key, and switch on *Notebook access*.
4. Pick a GPU: *Runtime > Change runtime type*.

**Then** use *Runtime > Run all*. The last cell prints an address and a key. Paste both into Deterministic Coder under *Settings > Local AI server / Colab* and tick the box "This server is on my network".

**Good to know**
- Keep this tab open. Colab ends idle or closed sessions and has a time limit per session, so you will have to run it again.
- Every session starts empty, so the model files download again unless you switch on the Google Drive cache in the settings.
- The Drive cache is **off by default**. When on, it uses your own Google account. A free Google account has 15 GB, shared with Gmail and Photos: neither 27B download can ever fit there, the 14B (about 10 GB) fits only if about 11 GB is free. The notebook checks first and, if it will not fit, says so and downloads each session instead.
- Unverified on a real runtime: vLLM with these repos on each GPU (and which vLLM version Qwen3.8 needs), `tailscale serve` accepting connections on Colab, the context sizes, tool calling, and the speed.

In [ ]:
#@title 1. Settings { display-mode: "form" }
TIER = "auto"                         #@param ["auto", "son", "father"]
MODEL = ""                            #@param {type:"string"}
SERVED_NAME = ""                      #@param {type:"string"}
THINKING = False                      #@param {type:"boolean"}
CONTEXT_TOKENS = 0                    #@param {type:"integer"}
ENFORCE_EAGER = False                 #@param {type:"boolean"}
PORT = 8000                           #@param {type:"integer"}
TAILSCALE_HOSTNAME = "colab-dc"       #@param {type:"string"}
USE_DRIVE_CACHE = False               #@param {type:"boolean"}
DRIVE_FOLDER = "DeterministicCoder"   #@param {type:"string"}
# TIER: auto = the GPU decides (see the table at the top). father = the 27B model (needs a GPU of at least 38 GB). son = the 14B 4-bit model (any GPU).
# MODEL: leave empty. Or type exactly one id from cell 2 (Qwen/Qwen3.8-27B, Qwen/Qwen3.8-27B-FP8, Qwen/Qwen2.5-Coder-14B-Instruct-AWQ) to force it;
#   a model that does not fit the GPU is refused. Other ids have no size table here.
# SERVED_NAME: the model name the app will send. Empty = a name derived from the model.
# THINKING: only the Qwen3.8 (father) model reasons first. False = direct answers (faster, what the patch engine wants).
# CONTEXT_TOKENS: 0 = the size chosen for your GPU in cell 2. Anything above the model's native length is refused (no YaRN).
# ENFORCE_EAGER: adds --enforce-eager. Leave False. Turn on only if vLLM fails while starting up (one vLLM recipe reports a CUDA-graph
#   startup failure without it: unverified for this notebook). It costs speed.
# USE_DRIVE_CACHE: False = download to this runtime's disk every session. True = keep a verified copy in YOUR Google Drive,
#   under MyDrive/<DRIVE_FOLDER>/models/ (and say so plainly if it will not fit).

In [ ]:
#@title 2. Models and sizes (this cell only defines the table; the arithmetic is in the comments) { display-mode: "form" }
import json

FATHER_BF16 = 'Qwen/Qwen3.8-27B'
FATHER_FP8 = 'Qwen/Qwen3.8-27B-FP8'
SON = 'Qwen/Qwen2.5-Coder-14B-Instruct-AWQ'

# Hugging Face API lookups, 2026-10-07: all three answered HTTP 200, public (not gated), Apache-2.0. Qwen/Qwen3.8-27B-AWQ answers
# HTTP 401 (there is no official AWQ build of the 27B model), so it is not used.
#   Qwen3.8-27B: architecture Qwen3_5ForConditionalGeneration, bf16, 55.56 GB of safetensors.
#   Qwen3.8-27B-FP8: same architecture, quant_method fp8 in its config.json, 30.89 GB in 81 files.
#   Qwen2.5-Coder-14B-Instruct-AWQ: Qwen2ForCausalLM, quant_method awq (4 bit, group 128), 9.98 GB, native context 32768.
#
# KV cache per token = (layers with full attention) x 2 (K and V) x kv_heads x head_dim x 2 bytes (fp16). From each repo's config.json:
#   Qwen3.8-27B (both repos): 16 of 64 layers are full attention (the others are linear attention), 4 kv heads, head_dim 256
#       -> 16 x 2 x 4 x 256 x 2 B = 64 KiB per token: 32,768 tokens = 2 GiB, 65,536 = 4 GiB, 131,072 = 8 GiB.
#       The linear-attention layers also keep a small fixed state per request that this ignores (unverified how much vLLM sets aside).
#   Qwen2.5-Coder-14B-AWQ: 48 layers, 8 kv heads, head_dim 128 (5120 / 40 heads)
#       -> 48 x 2 x 8 x 128 x 2 B = 192 KiB per token: 8,192 = 1.5 GiB, 16,384 = 3 GiB, 32,768 = 6 GiB.
# Usable memory = GPU memory x --gpu-memory-utilization (0.90), and vLLM needs room for activations and CUDA graphs. A flat 2 GiB is
# assumed for that here (a guess, unverified).
#   A100 / H100 80 GB (79.6 to 80 GiB): 0.90 x 79.6 = 71.6 GiB. bf16 weights 55.56 GB = 51.7 GiB -> 19.9 GiB left, minus 2 = 17.9 GiB.
#       131,072 tokens need 8 GiB. The native maximum is 262,144 (16 GiB for one full-length request); half of it is the cautious pick.
#   A100 40 GB (39.6 to 40 GiB): 0.90 x 39.6 = 35.6 GiB. bf16 does not fit (51.7 GiB). The FP8 repo is 30.89 GB = 28.8 GiB
#       -> 6.8 GiB left, minus 2 = 4.8 GiB. 32,768 tokens need 2 GiB. 65,536 would need 4 GiB: it fits on paper only barely, so it is unproven.
#   L4 24 GB (22.5 GiB): even the FP8 repo (28.8 GiB) does not fit, so the 27B model is out. 14B AWQ weights 9.98 GB = 9.3 GiB;
#       0.90 x 22.5 = 20.3 GiB -> 11.0 left, minus 2 = 9.0 GiB. 32,768 tokens (the model's native maximum) need 6 GiB.
#   T4 16 GB (15 GiB, nvidia-smi shows 15,360 MiB): 0.90 x 15 = 13.5 GiB -> 14B AWQ 9.3 GiB leaves 4.2, minus 2 = 2.2 GiB.
#       8,192 tokens need 1.5 GiB; 16,384 would need 3 GiB, more than the estimate, so it is not tried. Deliberately tight.
#       A T4 has no bf16, so the Son profile uses --dtype half (the AWQ repo is fp16 anyway).
# vLLM's quantization table lists AWQ as supported on Turing (T4); whether its attention backend works on a T4 is unverified.
# No --quantization flag is passed for either quantized repo: each repo's config.json names its method (fp8, awq) and vLLM reads it.
# The current vLLM docs spell the AWQ method auto_awq, so the older name awq is not forced. Whether auto-detection works on the
# installed vLLM version is unverified.
MODELS = {
    FATHER_BF16: {
        'tier': 'father', 'family': 'qwen3.8', 'served': 'qwen3.8-27b', 'native_ctx': 262144, 'dtype': None, 'bytes': 55563006776,
        'hint': f'Use {FATHER_FP8} on a 40 GB GPU.',
        'tiers': [(70, {'ctx': 131072, 'util': 0.90, 'note': 'bf16 weights'})],   # (minimum GPU GiB, settings)
    },
    FATHER_FP8: {
        'tier': 'father', 'family': 'qwen3.8', 'served': 'qwen3.8-27b-fp8', 'native_ctx': 262144, 'dtype': None, 'bytes': 30890049597,
        'hint': f'Use {SON} on a smaller GPU.',
        'tiers': [(70, {'ctx': 131072, 'util': 0.90, 'note': 'Qwen\'s 8-bit repo (80 GB GPU: you forced this model)'}),
                  (38, {'ctx': 32768, 'util': 0.90, 'note': 'Qwen\'s own 8-bit (fp8) repo, because bf16 does not fit'})],
    },
    SON: {
        'tier': 'son', 'family': 'qwen2.5-coder', 'served': 'qwen2.5-coder-14b-awq', 'native_ctx': 32768, 'dtype': 'half', 'bytes': 9980170584,
        'hint': '',
        'tiers': [(21, {'ctx': 32768, 'util': 0.90, 'note': '4-bit AWQ weights (L4 class or larger), unverified'}),
                  (0, {'ctx': 8192, 'util': 0.90, 'note': '4-bit AWQ weights (T4 class, free Colab), unverified'})],
    },
}

# vLLM flags per model family. The Qwen3 parsers belong to the Qwen3.8 (father) model only; Qwen2.5-Coder tool calls use the hermes format.
FAMILY_FLAGS = {
    'qwen3.8': ['--reasoning-parser', 'qwen3', '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder'],
    'qwen2.5-coder': ['--enable-auto-tool-choice', '--tool-call-parser', 'hermes'],
}


def pick_profile(vram_gib, model='', tier='auto', context_override=0):
    """Pure function: GPU memory in GiB (+ optional forced tier / model / context) -> settings dict. Raises ValueError with a plain message."""
    tier = (tier or 'auto').strip().lower()
    model = (model or '').strip()
    if tier not in ('auto', 'son', 'father'):
        raise ValueError('TIER must be auto, son or father.')
    if not model:
        if tier == 'auto':
            tier = 'father' if vram_gib >= MODELS[FATHER_FP8]['tiers'][-1][0] else 'son'
        model = SON if tier == 'son' else (FATHER_BF16 if vram_gib >= 70 else FATHER_FP8)
    if model not in MODELS:
        raise ValueError('MODEL must be empty or exactly one of: ' + ', '.join(MODELS) + '. Other models have no size table here.')
    m = MODELS[model]
    if tier not in ('auto', m['tier']):
        raise ValueError(f'TIER is {tier} but MODEL {model} belongs to tier {m["tier"]}. Change one of them.')
    chosen = next((t for floor, t in m['tiers'] if vram_gib >= floor), None)
    if chosen is None:
        raise ValueError(f'{model} needs a GPU with at least {m["tiers"][-1][0]} GB; this GPU has {vram_gib:.0f} GB. {m["hint"]} '
                         'Leave TIER on auto to let the GPU decide.')
    ctx = chosen['ctx']
    if context_override:
        if context_override > m['native_ctx']:
            raise ValueError(f'CONTEXT_TOKENS {context_override} is above the {m["native_ctx"]} tokens this model supports natively '
                             '(this notebook does not enable YaRN).')
        ctx = int(context_override)
    return {'model': model, 'tier': m['tier'], 'family': m['family'], 'dtype': m['dtype'], 'ctx': ctx, 'util': chosen['util'],
            'note': chosen['note'], 'served': m['served'], 'verified': False}


def serve_args(profile, thinking=False, enforce_eager=False):
    """vLLM flags that depend on the profile (model family), never global."""
    a = ['--max-model-len', str(profile['ctx']), '--gpu-memory-utilization', str(profile['util'])]
    if profile['dtype']:
        a += ['--dtype', profile['dtype']]
    if enforce_eager:
        a += ['--enforce-eager']
    a += FAMILY_FLAGS[profile['family']]
    if profile['family'] == 'qwen3.8' and not thinking:
        a += ['--default-chat-template-kwargs', json.dumps({'enable_thinking': False})]
    return a

In [ ]:
#@title 3. Check the GPU and pick a model { display-mode: "form" }
import subprocess

try:
    first = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], text=True).strip().splitlines()[0]
except Exception:
    raise SystemExit('No GPU found. Runtime > Change runtime type > choose a GPU (T4 is the free one; L4, A100 and H100 need Colab Pro or Pro+), then run this cell again.')
gpu_name, mem_mb = [s.strip() for s in first.split(',')]
vram_gib = int(mem_mb) / 1024

try:
    profile = pick_profile(vram_gib, MODEL, TIER, int(CONTEXT_TOKENS or 0))
except ValueError as e:
    raise SystemExit(f'{gpu_name} ({vram_gib:.0f} GB): {e}')
SERVED = SERVED_NAME.strip() or profile['served']
print(f"GPU: {gpu_name}, {vram_gib:.0f} GB  ->  tier {profile['tier']}: {profile['model']}, {profile['note']}, context {profile['ctx']} tokens")
if profile['tier'] == 'son' and vram_gib < MODELS[FATHER_FP8]['tiers'][-1][0]:
    print(f'The 27B model needs a GPU of at least {MODELS[FATHER_FP8]["tiers"][-1][0]} GB; this one cannot hold it, so the smaller 14B 4-bit model is used.')
if vram_gib < 21:
    print('Free-tier (T4 class) path: UNVERIFIED. It has not been run; do not expect it to be fast or even to start.')

In [ ]:
#@title 4. Install the model server (several minutes the first time) { display-mode: "form" }
import os, subprocess, sys

def sh(cmd, **kw):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True, **kw)

# vLLM goes into its own virtual environment, so it cannot disturb Colab's own packages.
# The newest vLLM is installed, not a pinned one. Which vLLM version Qwen3.8 (architecture Qwen3_5ForConditionalGeneration) needs is
# UNVERIFIED: if cell 7 reports an unsupported architecture, that is the first thing to look at (the vLLM log tail is printed).
sh(f'{sys.executable} -m pip install -q uv huggingface_hub')
sh('uv venv /content/vllm-env')
sh('uv pip install --python /content/vllm-env/bin/python -U vllm')
VLLM = '/content/vllm-env/bin/vllm'
print(subprocess.run([VLLM, '--version'], capture_output=True, text=True).stdout.strip() or 'vllm installed')

os.environ['HF_HOME'] = '/content/hf-cache'      # scratch space on this runtime; the model files themselves go where cell 5 says
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # optional: only helps if Hugging Face rate-limits you
except Exception:
    pass

In [ ]:
#@title 5. Get the model files (optional Google Drive copy) { display-mode: "form" }
import json, os, shutil, time

GIB = 2 ** 30
MARGIN = 2 * GIB          # allowance for tokenizer/config files and filesystem overhead (a rough number, not measured)
MANIFEST = 'dc-manifest.json'

def repo_files(model, token=''):
    """{file name: exact size in bytes} for every file in the repo, from the Hugging Face API. {} if it cannot be read."""
    try:
        from huggingface_hub import HfApi
        info = HfApi().model_info(model, files_metadata=True, token=token or None)
        return {s.rfilename: s.size for s in info.siblings}
    except Exception as e:
        print('Could not read the file list from Hugging Face (' + type(e).__name__ + ').')
        return {}

def dir_bytes(folder):
    return sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(folder) for f in fs) if os.path.isdir(folder) else 0

def disk_free(path):
    try:
        return shutil.disk_usage(path).free
    except Exception:
        return None

def space_problem(need, free, where):
    """None if `need` bytes (plus a margin) fit in `free` bytes, else a plain sentence saying why it does not."""
    if free is None:
        return f'Cannot read the free space on {where}.'
    if need + MARGIN > free:
        return (f'Not enough room on {where}: needs {need / GIB:.1f} GiB (plus {MARGIN / GIB:.0f} GiB of slack), '
                f'only {free / GIB:.1f} GiB is free.')
    return None

def cache_complete(folder, expected):
    """expected = {file name: exact size} for the whole repo. True only if the manifest exists AND lists exactly those files and sizes
    AND every file is on disk with exactly that size AND every shard named in model.safetensors.index.json is among them.
    A half-copied folder never passes: the manifest is written last, after the sizes were checked."""
    try:
        with open(os.path.join(folder, MANIFEST)) as f:
            listed = json.load(f)['files']
        index = os.path.join(folder, 'model.safetensors.index.json')
        shards = set()
        if os.path.isfile(index):
            with open(index) as f:
                shards = set(json.load(f)['weight_map'].values())
    except Exception:
        return False
    if not expected or listed != expected or not shards <= set(expected):
        return False
    return all(os.path.isfile(os.path.join(folder, n)) and os.path.getsize(os.path.join(folder, n)) == size for n, size in expected.items())

def write_manifest(folder, expected, model=''):
    """Verify every file's exact size, then write the manifest (last, atomically). Raises, and writes nothing, if any file is missing or wrong."""
    for name, size in expected.items():
        path = os.path.join(folder, name)
        if not os.path.isfile(path) or os.path.getsize(path) != size:
            raise RuntimeError(f'{name} is missing or has the wrong size; not marking this copy complete.')
    tmp = os.path.join(folder, MANIFEST + '.tmp')
    with open(tmp, 'w') as f:
        json.dump({'repo': model, 'files': expected}, f)
    os.replace(tmp, os.path.join(folder, MANIFEST))

def download(model, expected, final, token=''):
    """Hugging Face -> <final>.partial (snapshot_download resumes after a break), verify, manifest last, then rename to <final>."""
    from huggingface_hub import snapshot_download
    partial = final + '.partial'
    os.makedirs(partial, exist_ok=True)
    snapshot_download(repo_id=model, local_dir=partial, token=token or None)
    write_manifest(partial, expected, model)
    shutil.rmtree(final, ignore_errors=True)
    os.replace(partial, final)

def copy_verified(src, final, expected, model=''):
    """Copy a folder (either direction, Drive <-> runtime disk) to <final>.partial, verify, manifest last, then rename.
    A leftover .partial is deleted first, never reused."""
    partial = final + '.partial'
    shutil.rmtree(partial, ignore_errors=True)
    shutil.copytree(src, partial, ignore=shutil.ignore_patterns('.cache', MANIFEST))
    write_manifest(partial, expected, model)
    shutil.rmtree(final, ignore_errors=True)
    os.replace(partial, final)

def rate(nbytes, secs):
    return f'{nbytes / 1e6 / max(secs, 1e-6):.0f} MB/s'

# ---- run (the test suite only reads the lines above this one) ----
MODEL_ID = profile['model']
SAFE = MODEL_ID.replace('/', '-')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
LOCAL_ROOT = '/content/models'
os.makedirs(LOCAL_ROOT, exist_ok=True)
MODEL_PATH = os.path.join(LOCAL_ROOT, SAFE)          # vLLM always loads from this local copy
expected = repo_files(MODEL_ID, HF_TOKEN)
need = sum(s for s in expected.values() if s) or MODELS[MODEL_ID]['bytes']     # fallback: the weights size measured on 2026-10-07
print(f'{MODEL_ID}: about {need / GIB:.1f} GiB to get.')
cache = {'state': 'off', 'path': '', 'files': len(expected), 'gb': round(need / 1e9, 2), 'drive_free_gb': None, 'served_from': MODEL_PATH}

local_have = cache_complete(MODEL_PATH, expected)
if not expected:
    problem = space_problem(need, disk_free(LOCAL_ROOT), 'the runtime disk')
    raise SystemExit((problem + '\n' if problem else '') + 'Cannot read the exact file list from Hugging Face, so the download could not be verified. Try again in a minute.')

# --- Google Drive (optional): decide first, so a "will not fit" shows up before any big download ---
drive_final, drive_have, drive_room = '', False, False
if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')          # YOUR Google Drive; Colab asks you to sign in and allow it. Saying no is fine: the notebook goes on without the cache.
        folder = os.path.basename(DRIVE_FOLDER.strip().rstrip('/')) or 'DeterministicCoder'
        drive_models = os.path.join('/content/drive/MyDrive', folder, 'models')
        os.makedirs(drive_models, exist_ok=True)
        drive_final = os.path.join(drive_models, SAFE)
        cache['path'] = drive_final
        shutil.rmtree(drive_final + '.partial', ignore_errors=True)       # a leftover .partial is never reused
        drive_have = cache_complete(drive_final, expected)
        free = disk_free(drive_models)
        cache['drive_free_gb'] = None if free is None else round(free / 1e9, 2)
        if not drive_have:
            problem = space_problem(need, free, 'your Google Drive')
            if problem:
                cache['state'] = 'skipped-no-space'
                print('Drive cache skipped:', problem, 'Downloads each session. (A free Google account has 15 GB shared with Gmail and Photos.)')
            else:
                drive_room = True
    except Exception as e:
        cache['state'] = 'skipped-mount-failed'
        print('Drive cache skipped: could not mount or use Google Drive (' + type(e).__name__ + '). Continuing without it.')

# --- get the files onto the runtime disk (local first) ---
if not local_have:
    if drive_have:
        try:
            t = time.time()
            copy_verified(drive_final, MODEL_PATH, expected, MODEL_ID)
            print(f'Copied from Drive to the runtime disk: {rate(need, time.time() - t)}.')
        except Exception as e:
            print('Copy from Drive failed (' + str(e)[:200] + '). Downloading from Hugging Face instead.')
            cache['state'] = 'copy-failed'
            drive_have = False
    if not cache_complete(MODEL_PATH, expected):
        problem = space_problem(need - dir_bytes(MODEL_PATH + '.partial'), disk_free(LOCAL_ROOT), 'the runtime disk')
        if problem:
            raise SystemExit(problem + '\nChoose a smaller model (TIER = son) or a runtime with a bigger disk.')
        t = time.time()
        download(MODEL_ID, expected, MODEL_PATH, HF_TOKEN)
        print(f'Downloaded from Hugging Face: {rate(need, time.time() - t)}.')

# --- keep a verified copy on Drive for next time ---
if drive_have:
    cache['state'] = 'reused'
elif drive_room:
    try:
        t = time.time()
        copy_verified(MODEL_PATH, drive_final, expected, MODEL_ID)
        cache['state'] = 'created'
        print(f'Copied to Google Drive: {rate(need, time.time() - t)} (speeds vary; this does not show Drive is faster than Hugging Face).')
    except Exception as e:
        cache['state'] = 'copy-failed'
        shutil.rmtree(drive_final + '.partial', ignore_errors=True)
        print('Copy to Drive failed (' + str(e)[:200] + '). Nothing incomplete is kept; the model still runs from the runtime disk.')
WEIGHTS_FROM = f'the runtime disk ({MODEL_PATH})'
print('Model files will be loaded from:', MODEL_PATH, '| Drive cache:', cache['state'])

In [ ]:
#@title 6. Join your Tailscale network { display-mode: "form" }
import json, os, subprocess, time

try:
    from google.colab import userdata
    TS_AUTHKEY = userdata.get('TS_AUTHKEY')
except Exception:
    TS_AUTHKEY = ''
if not TS_AUTHKEY:
    raise SystemExit('Add a Colab secret named TS_AUTHKEY (key icon in the left bar), switch on Notebook access, and run this cell again.')

TS_SOCK = '/tmp/tailscaled.sock'
ts = lambda *a: subprocess.run(['tailscale', f'--socket={TS_SOCK}', *a], capture_output=True, text=True)

if subprocess.run('command -v tailscale', shell=True, capture_output=True).returncode != 0:
    subprocess.run('curl -fsSL https://tailscale.com/install.sh | sh', shell=True, check=True)

# Colab has no network device to give Tailscale, so it runs in userspace-networking mode, with state only in memory.
# UNVERIFIED on Colab: that `tailscale serve` below really accepts inbound connections from your other devices in this mode.
tailscaled = subprocess.Popen(['tailscaled', '--tun=userspace-networking', '--state=mem:', f'--socket={TS_SOCK}'],
                              stdout=open('/content/tailscaled.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(3)
up = ts('up', f'--authkey={TS_AUTHKEY}', f'--hostname={TAILSCALE_HOSTNAME}', '--accept-dns=false')
if up.returncode != 0:
    raise SystemExit('tailscale up failed:\n' + (up.stderr or up.stdout))
# In this mode the node has no network interface, so `serve` is what hands incoming tailnet connections to the local port.
srv = ts('serve', '--bg', f'--tcp={PORT}', f'tcp://127.0.0.1:{PORT}')
if srv.returncode != 0:
    raise SystemExit('tailscale serve failed:\n' + (srv.stderr or srv.stdout))
me = json.loads(ts('status', '--json').stdout)['Self']
TS_NAME = (me.get('DNSName') or '').rstrip('.')
TS_IP = (me.get('TailscaleIPs') or [''])[0]
print('On your tailnet as', TS_NAME or TS_IP, f'({TS_IP})')

In [ ]:
#@title 7. Start the model and print your address { display-mode: "form" }
import json, secrets, subprocess, time, urllib.request, urllib.error

API_KEY = 'dc-' + secrets.token_urlsafe(24)     # made fresh on every run; it exists only in this session
cmd = [VLLM, 'serve', MODEL_PATH,
       '--host', '127.0.0.1', '--port', str(PORT), '--api-key', API_KEY,
       '--served-model-name', SERVED] + serve_args(profile, THINKING, ENFORCE_EAGER)     # flags come from the profile (cell 2)
print('vLLM flags:', ' '.join(a for a in cmd if a != API_KEY))
logf = open('/content/vllm.log', 'w')
vllm_proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)

def call(path, body=None, timeout=60):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}{path}', data=json.dumps(body).encode() if body else None,
                                 headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def log_tail(n=40):
    return '\n'.join(open('/content/vllm.log', errors='replace').read().splitlines()[-n:])

print('Loading the model. This can take several minutes.')
t0 = time.time()
while True:
    if vllm_proc.poll() is not None:
        raise SystemExit('The model server stopped. Last log lines:\n' + log_tail())
    try:
        call('/v1/models', timeout=5)
        break
    except Exception:
        time.sleep(10)
    if time.time() - t0 > 40 * 60:
        raise SystemExit('Still not ready after 40 minutes. Last log lines:\n' + log_tail())
print(f'Ready after {(time.time() - t0) / 60:.1f} minutes.')

# One small request that offers a tool, so the speed and the tool-call path are measured, not guessed.
TOOLS = [{'type': 'function', 'function': {'name': 'add_numbers', 'description': 'Add two integers.',
          'parameters': {'type': 'object', 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b']}}}]
t1 = time.time()
try:
    reply = call('/v1/chat/completions', {'model': SERVED, 'max_tokens': 1024 if THINKING else 256, 'temperature': 0, 'tools': TOOLS, 'tool_choice': 'auto',
                                          'messages': [{'role': 'user', 'content': 'Use the add_numbers tool to add 17 and 25.'}]}, timeout=300)
except Exception as e:
    raise SystemExit('The test request failed (' + type(e).__name__ + '). Last log lines:\n' + log_tail())
secs = time.time() - t1
toks = reply.get('usage', {}).get('completion_tokens', 0)
print(f'Test request: {toks} tokens in {secs:.1f} s ({toks / secs:.1f} tokens/s, includes the first-request warm-up).')
calls = (reply['choices'][0]['message'].get('tool_calls') or [])
if calls:
    print('Tool call came back:', calls[0]['function']['name'], calls[0]['function']['arguments'])
else:
    print('No tool call came back (the model answered in plain text). Tool use may not work with these flags; send this output back.')

host = TS_NAME or TS_IP
print('\n' + '=' * 64)
print('Paste these into Deterministic Coder (Settings > Local AI server / Colab):')
print(f'  Address : http://{host}:{PORT}/v1')
print(f'  Key     : {API_KEY}')
print(f'  Model   : {SERVED}')
print(f'  Context : {profile["ctx"]}')
print(f'  Tier    : {profile["tier"]} ({profile["model"]}, unverified)')
print(f'  Cache   : Drive cache {cache["state"]}; weights loaded from {WEIGHTS_FROM}')
print('  Tick the box "This server is on my network". Do not share this address or key.')
print('=' * 64)
print('DC-CACHE ' + json.dumps(cache))     # one machine-readable line: state, path, files, GB, Drive free GB, served-from

In [ ]:
#@title 8. Keep it running (leave this cell running; stop it to shut everything down) { display-mode: "form" }
import time

print('Running. Stop this cell (the square button) to shut the server down.')
try:
    while True:
        time.sleep(60)
        if vllm_proc.poll() is not None:
            print('The model server stopped. Last log lines:\n' + log_tail())
            break
        if tailscaled.poll() is not None:
            print('Tailscale stopped. Re-run cell 6, then this cell.')
            break
except KeyboardInterrupt:
    pass
finally:
    for p in (vllm_proc,):
        if p.poll() is None:
            p.terminate()
    ts('down')
    print('Shut down.')